# GameTheory-6d : Statique comparative — séparer sympathie et engagement par les gains d'autrui

GT-06c (section 7) retire la capacité de punir et classe la coopération résiduelle en trois : menace folk, utilité transformée (sympathie), reste (candidat engagement au sens de Sen). Mais son classifieur rend un verdict qui **dépend du choix d'alpha par l'analyste** : pour une même trajectoire, `alpha = 1.0` conclut « sympathie » et `alpha = 0` conclut « engagement ». Le seuil `alpha* = (T-R)/(R-S)` n'est pas un seuil de diagnostic — c'est une **frontière d'indiscernabilité**.

Ce notebook rend la question **identifiable** : les deux mécanismes ne dépendent pas des mêmes quantités. La sympathie fait entrer le bien-être d'autrui dans mon utilité — ma coopération doit **répondre à l'amplitude des gains d'autrui**. L'engagement suit une règle contre mon classement de bien-être — ma coopération doit être **insensible** à ce que l'autre gagne. En faisant varier **les gains d'autrui seuls**, à gains propres strictement fixes, la pente du taux de coopération devient une **quantité mesurée**, et `alpha` cesse d'être un paramètre libre : il se dérive de la pente.

Références : Sen (1977), *Rational Fools* — sympathie vs engagement ; GT-06c §7 (le banc sans punition) ; issue #13042.

## Objectifs d'apprentissage

1. **Diagnostiquer l'indiscernabilité** : montrer pourquoi, sur des trajectoires de coopération seules, sympathie et engagement produisent les mêmes observations pour un choix adapté d'alpha.
2. **Construire la statique comparative** : une grille de jeux sans punition où **seule** la colonne des gains d'autrui varie — la ligne du joueur observé reste byte-identique d'une cellule à l'autre.
3. **Mesurer une pente** : taux de coopération par cellule, sur trajectoires reproductibles multi-graines, avec incertitude.
4. **Estimer alpha depuis la pente** — et le **valider par contrôles négatifs** : un agent pur-sympathie à alpha connu (l'estimateur doit le retrouver) et un agent pur-engagement à règle fixe (l'estimateur doit rendre une pente nulle).
5. **Admettre le verdict négatif** : une pente indiscernable de zéro avec contrôles fonctionnels se publie comme telle — « non identifié à ce niveau de bruit ».

## 1. Le banc sans punition et le problème d'identification

Le régime est celui de GT-06c §7a : l'adversaire est **sans mémoire et coopère quoi qu'il arrive**. Aucune séquence de mes coups ne peut être punie — la déviation n'a aucune conséquence future. Dans ce régime, coopérer est **contre-intérêt** au sens de l'utilité propre : `V_dev - V_coop = (T - R)/(1-delta) > 0`.

Le dilemme du prisonnier est celui de GT-06c (convention Axelrod) : `T=5, R=3, P=1, S=0`. Le seuil de sympathie du classifieur vaut `alpha* = (T-R)/(R-S) = 2/3` : c'est la sympathie qui **annule exactement** l'écart d'intérêt dans le jeu symétrique. Mais ce seuil n'identifie rien : il sépare deux **lectures** d'une même trajectoire, pas deux **comportements**.

In [1]:
import numpy as np

# === Parametres du banc (byte-identiques a GT-06c) ===
T, R, P, S = 5.0, 3.0, 1.0, 0.0   # convention Axelrod, ligne du joueur observe
DELTA = 0.6                        # patience (inerte ici : adversaire sans memoire)

# Le joueur observe fait face a un adversaire SANS MEMOIRE qui coopere toujours.
# Sa colonne de gains a deux entrees : ce que touche autrui en (C,C) -> R_autre,
# ce qu'il touche en (D,C) -> S_autre. La LIGNE du joueur observe (T, R, P, S)
# ne change JAMAIS dans ce notebook.
S_AUTRE = 0.0                      # colonne d'autrui quand je defaute (fixee)

alpha_etoile = (T - R) / (R - S)
ecart_pur = (T - R) / (1 - DELTA)  # ecart d'interet actualise, utilite propre
print(f"alpha* = (T-R)/(R-S) = {alpha_etoile:.3f}  (frontiere d'indiscernabilite, pas un diagnostic)")
print(f"Ecart d'interet (utilite propre, actualise) : V_dev - V_coop = {ecart_pur:.2f} > 0")
print("Cooperation observee dans ce regime = contre-interet : le residu demande un mecanisme.")

alpha* = (T-R)/(R-S) = 0.667  (frontiere d'indiscernabilite, pas un diagnostic)
Ecart d'interet (utilite propre, actualise) : V_dev - V_coop = 5.00 > 0
Cooperation observee dans ce regime = contre-interet : le residu demande un mecanisme.


## 2. La grille de statique comparative

**Le geste expérimental** : faire varier `R_autre` — ce que l'autre gagne quand je coopère — sur une plage qui couvre tout l'espace utile, **à ligne du joueur observé inchangée**. Si un mécanisme de sympathie porte la coopération (`U = u_mien + alpha * u_autrui`), coopérer devient d'autant plus attrayant que l'autre en tire parti : le taux doit **monter** avec `R_autre`. Si un engagement porte la coopération (une règle suivie contre mon classement de bien-être), rien ne bouge : le taux doit être **plat**.

La condition qui rend la statique interprétable — gains propres strictement identiques d'une cellule à l'autre — se **vérifie** et se **dit**, elle ne se suppose pas.

In [2]:
# === Grille : seule la colonne d'autrui varie ===
R_AUTRE_GRID = np.linspace(0.0, 6.0, 13)   # inclut 3.0 = jeu symetrique de GT-06c
N_SEEDS = 5
SEEDS = [0, 1, 7, 42, 99]                  # multi-graine >= 4
HORIZON = 200                              # tours par trajectoire

def ligne_joueur():
    """Ligne de gains du joueur observe — constant sur TOUTE la grille."""
    return {"C": {"C": R, "D": T}, "D": {"C": P, "D": S}}  # adversaire always-C : seuls (C,C)/(D,C) sont joues

# Verification byte-identique de la ligne (condition d'interpretation)
ligne_ref = ligne_joueur()
for r_a in R_AUTRE_GRID:
    assert ligne_joueur() == ligne_ref, f"ligne corrompue a R_autre={r_a}"
print(f"Grille : R_autre de {R_AUTRE_GRID[0]:.1f} a {R_AUTRE_GRID[-1]:.1f} en {len(R_AUTRE_GRID)} cellules, S_autre = {S_AUTRE:.1f} fixe")
print(f"Ligne du joueur observe (T,R,P,S) = ({T},{R},{P},{S}) — verifiee byte-identique sur les {len(R_AUTRE_GRID)} cellules")
print(f"Trajectoires : {N_SEEDS} graines x {HORIZON} tours = {N_SEEDS * HORIZON} decisions par cellule")

Grille : R_autre de 0.0 a 6.0 en 13 cellules, S_autre = 0.0 fixe
Ligne du joueur observe (T,R,P,S) = (5.0,3.0,1.0,0.0) — verifiee byte-identique sur les 13 cellules
Trajectoires : 5 graines x 200 tours = 1000 decisions par cellule


## 3. Le sujet : un agent à sympathie cachée

Le joueur observé choisit ses coups par **logit (softmax)** sur une utilité transformée : `U(a) = u_mien(a) + alpha * u_autrui(a)`, avec probabilité de coopérer `p(C) = sigma(beta * [U(C) - U(D)])`. Son `alpha` est **caché** — c'est précisément ce que l'expérience prétend mesurer. L'adversaire étant sans mémoire et toujours coopératif, chaque tour est un tirage indépendant sur le même logit : le taux de coopération d'une trajectoire est un estimateur binomial de `p(C)`.

Attention à ce que le modèle implique : `p(C) = sigma(beta * [(R - T) + alpha * (R_autre - S_autre)])`. La dépendance en `R_autre` est **linéaire dans le logit**, de pente `beta * alpha`. C'est cette pente — mesurée sur les taux — qui porte l'identification.

In [3]:
# === Sujet : agent logit a sympathie cachee ===
ALPHA_VRAI = 0.45      # < alpha* : au jeu symetrique, sa sympathie ne suffit pas a coopere majoritairement
BETA = 1.0             # temperature inverse declaree de l'estimateur

def u_autrui(action, R_autre):
    """Gain d'autrui quand le joueur observe joue <action> (adversaire always-C)."""
    return R_autre if action == "C" else S_AUTRE

def taux_cooperation(alpha, R_autre, seed, beta=BETA, horizon=HORIZON):
    """Une trajectoire : taux de cooperation d'un agent logit a sympathie alpha."""
    rng = np.random.default_rng(seed)
    dU = (R - T) + alpha * (R_autre - S_AUTRE)   # U(C) - U(D) dans le logit
    p_c = 1.0 / (1.0 + np.exp(-beta * dU))
    tirages = rng.random(horizon)
    return float(np.mean(tirages < p_c))

# Mesure : taux moyen par cellule, multi-graines
taux_sujet = np.array([[taux_cooperation(ALPHA_VRAI, r_a, s) for s in SEEDS] for r_a in R_AUTRE_GRID])
taux_sujet_moy = taux_sujet.mean(axis=1)

print(f"alpha cache du sujet = {ALPHA_VRAI}   (l'estimateur ne le connait pas)")
print(f"{'R_autre':>7} | {'taux C':>6} | theorique sigma(dU)")
print("-" * 42)
for r_a, t in zip(R_AUTRE_GRID, taux_sujet_moy):
    dU = (R - T) + ALPHA_VRAI * (r_a - S_AUTRE)
    print(f"{r_a:7.1f} | {t:6.3f} |   {1/(1+np.exp(-BETA*dU)):.3f}")

alpha cache du sujet = 0.45   (l'estimateur ne le connait pas)
R_autre | taux C | theorique sigma(dU)
------------------------------------------
    0.0 |  0.102 |   0.119
    0.5 |  0.127 |   0.145
    1.0 |  0.152 |   0.175
    1.5 |  0.185 |   0.210
    2.0 |  0.220 |   0.250
    2.5 |  0.274 |   0.294
    3.0 |  0.320 |   0.343
    3.5 |  0.375 |   0.395
    4.0 |  0.434 |   0.450
    4.5 |  0.485 |   0.506
    5.0 |  0.545 |   0.562
    5.5 |  0.602 |   0.617
    6.0 |  0.655 |   0.668


**Lecture de la sortie committée** : le taux monte régulièrement de ~0,12 (`R_autre = 0` : coopérer ne profite à personne) à ~0,67 (`R_autre = 6` : coopérer enrichit fortement autrui) en passant par ~0,34 au jeu symétrique. La courbe est une **sigmoïde** en `R_autre` — c'est la signature d'un mécanisme dont l'utilité intègre le bien-être d'autrui. Un agent à engagement pur, lui, tracerait une ligne horizontale : la différence n'est pas une nuance d'interprétation, c'est une **propriété observable**. L'écart entre taux mesuré et théorique (`sigma(dU)` calculée avec le alpha caché) reste de l'ordre du bruit binomial — la simulation fait ce qu'elle prétend faire.

## 4. L'estimateur : alpha dérivé de la pente

On ajuste une régression logistique des taux mesurés contre `R_autre` : `p(C) = sigma(k * R_autre + b)`. Le modèle du sujet prédit `k = beta * alpha` et `b = beta * (R - T - alpha * S_autre)`. Avec la température `beta` déclarée, l'estimateur rend **alpha depuis la pente** : `alpha_hat = k / beta`. L'incertitude vient d'un **bootstrap sur les graines** — on ré-échantillonne les 5 graines avec remise, on ré-ajuste, et on lit l'intervalle à 95 %.

In [4]:
# === Estimateur : regression logistique (IRLS numpy) + bootstrap sur graines ===
def ajustement_logit(x, t, n):
    """IRLS : p = sigma(k*x + b), ponderes par le nombre de decisions n par point."""
    X = np.column_stack([x, np.ones_like(x)])
    theta = np.zeros(2)
    for _ in range(100):
        p = 1.0 / (1.0 + np.exp(-X @ theta))
        W = n * p * (1 - p) + 1e-9
        z = X @ theta + (t - p) / np.maximum(W / n, 1e-9)
        theta = np.linalg.solve(X.T @ (W[:, None] * X) + 1e-8 * np.eye(2), X.T @ (W * z))
    return theta  # [k, b]

def alpha_depuis_pente(taux_par_graine, beta=BETA):
    """alpha_hat (pente/beta) + IC95 bootstrap sur les graines."""
    n_dec = HORIZON  # par graine et cellule
    rng = np.random.default_rng(2024)
    boots = []
    for _ in range(2000):
        idx = rng.integers(0, len(SEEDS), size=len(SEEDS))
        t = taux_par_graine[:, idx].mean(axis=1)
        k, _ = ajustement_logit(R_AUTRE_GRID, t, n_dec)
        boots.append(k / beta)
    k_pointu, _ = ajustement_logit(R_AUTRE_GRID, taux_par_graine.mean(axis=1), n_dec)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return k_pointu / beta, lo, hi

alpha_hat, lo95, hi95 = alpha_depuis_pente(taux_sujet)
identifie = not (lo95 <= 0.0 <= hi95)
print(f"Sujet    : alpha_hat = {alpha_hat:.3f}   IC95 bootstrap = [{lo95:.3f}, {hi95:.3f}]")
print(f"           alpha cache (pour controle a posteriori seulement) = {ALPHA_VRAI}")
print(f"           pente significativement non nulle : {identifie}"
      f"  -> {'SYMPATHIE IDENTIFIEE (repond aux gains d autrui)' if identifie else 'NON IDENTIFIE a ce niveau de bruit'}")

Sujet    : alpha_hat = 0.472   IC95 bootstrap = [0.439, 0.506]
           alpha cache (pour controle a posteriori seulement) = 0.45
           pente significativement non nulle : True  -> SYMPATHIE IDENTIFIEE (repond aux gains d autrui)


**Lecture de la sortie committée** : l'estimateur, qui ne connaît ni `ALPHA_VRAI` ni le mécanisme, rend `alpha_hat` proche de la valeur cachée avec un intervalle de confiance qui exclut zéro. La quantité `alpha* = 0,667` de GT-06c ne joue **aucun rôle** ici : elle séparait deux lectures d'une trajectoire ; la pente, elle, sépare deux **comportements**. La précision vient du nombre de décisions par cellule (5 graines × 200 tours = 1000 tirages) — une pente estimée sur un tirage unique ne serait pas une pente.

## 5. Contrôles négatifs — la preuve que l'estimateur sait séparer

Une pente mesurée ne prouve pas que l'estimateur sait séparer : elle prouve seulement qu'il rend un nombre. Les contrôles installent la **vraie preuve** :

- **Contrôle sympathie** : un agent pur-sympathie à `alpha` **connu** (0,80), passé dans le même estimateur. Il doit rendre `alpha_hat ≈ 0,80` — l'estimateur retrouve ce qu'on y a mis.
- **Contrôle engagement** : un agent à **règle fixe insensible** — il coopère par conformité à la règle dans 95 % des tours, et tire uniformément sinon. Son taux est constant par construction, quel que soit `R_autre`. L'estimateur doit rendre une pente **nulle** (intervalle contenant 0).

Si le premier contrôle échoue, l'estimateur est biaisé ; si le second échoue, il hallucine une sensibilité qui n'existe pas. Le verdict sur le sujet n'a de valeur qu'avec les deux verts.

In [5]:
# === Controles negatifs ===
ALPHA_SYMPATHIE = 0.80   # connu, > alpha* : coopere majoritairement des le jeu symetrique
CONFORMITE_ENGAGEMENT = 0.95   # la regle est suivie a 95 %, tirage uniforme sinon

def taux_engagement(R_autre, seed, horizon=HORIZON):
    """Agent a regle fixe : insensible par construction aux gains d'autrui.

    Chaque cellule de la grille a SA PROPRE trajectoire (graine [seed, cellule]) :
    une vraie experience ne rejoue pas les memes tirages d'une cellule a l'autre.
    Le taux fluctue donc autour de la conformite par bruit binomial — et reste plat."""
    rng = np.random.default_rng([seed, int(round(R_autre * 1000))])
    conforme = rng.random(horizon) < CONFORMITE_ENGAGEMENT
    uniforme = rng.integers(0, 2, size=horizon)          # 1 = cooperer
    coopere = np.where(conforme, 1, uniforme)
    return float(np.mean(coopere))                        # R_autre volontairement ignore

taux_sympathie = np.array([[taux_cooperation(ALPHA_SYMPATHIE, r_a, s) for s in SEEDS] for r_a in R_AUTRE_GRID])
taux_eng = np.array([[taux_engagement(r_a, s) for s in SEEDS] for r_a in R_AUTRE_GRID])

a_symp, lo_s, hi_s = alpha_depuis_pente(taux_sympathie)
a_eng, lo_e, hi_e = alpha_depuis_pente(taux_eng)
ok_symp = abs(a_symp - ALPHA_SYMPATHIE) < 0.10
ok_eng = (lo_e <= 0.0 <= hi_e)

print(f"Controle sympathie (alpha connu = {ALPHA_SYMPATHIE}) : alpha_hat = {a_symp:.3f}  IC95 [{lo_s:.3f}, {hi_s:.3f}]"
      f"   -> {'RETOUVE' if ok_symp else 'ECHEC (estimateur biaise)'}")
print(f"Controle engagement (regle fixe)                 : alpha_hat = {a_eng:.3f}  IC95 [{lo_e:.3f}, {hi_e:.3f}]"
      f"   -> {'PENTE NULLE' if ok_eng else 'ECHEC (sensibilite hallucinee)'}")
print(f"Taux engagement moyens (min..max sur la grille) : {taux_eng.mean(axis=1).min():.3f}..{taux_eng.mean(axis=1).max():.3f} — plat par construction")

Controle sympathie (alpha connu = 0.8) : alpha_hat = 0.820  IC95 [0.765, 0.872]   -> RETOUVE
Controle engagement (regle fixe)                 : alpha_hat = -0.004  IC95 [-0.042, 0.034]   -> PENTE NULLE
Taux engagement moyens (min..max sur la grille) : 0.966..0.981 — plat par construction


**Lecture de la sortie committée** : les deux contrôles sont verts. L'estimateur retrouve l'alpha connu du pur-sympathie (au bruit d'échantillonnage près) et rend une pente statistiquement nulle sur l'agent à règle fixe — dont le taux, ~0,975 partout, est bien **plat** : il ne bouge pas quand `R_autre` double. C'est cette double preuve qui transforme le `alpha_hat` du sujet en **mesure** plutôt qu'en numéro de téléphone : le même pipeline, nourri d'un comportement insensible, rend zéro.

## 6. Verdict et borne d'indiscernabilité

La statique comparative produit son verdict en trois lignes — sujet, contrôle sympathie, contrôle engagement — et chaque verdict est **honnête par construction** : une pente indiscernable de zéro **avec contrôles fonctionnels** se publie comme telle. Le mauvais résultat serait de choisir un alpha et de rapporter le verdict qu'il implique : c'est précisément le geste que ce notebook retire à l'analyste.

In [6]:
# === Verdict final ===
lignes = [
    ("Sujet (alpha cache)",        alpha_hat, lo95, hi95),
    ("Controle sympathie connue",  a_symp,   lo_s,  hi_s),
    ("Controle engagement (regle)", a_eng,   lo_e,  hi_e),
]
print(f"{'Agent':<28} {'alpha_hat':>9} {'IC95':>18}  Verdict")
print("-" * 78)
for nom, a, lo, hi in lignes:
    plat = (lo <= 0.0 <= hi)
    verdict = ("pente nulle : INSENSIBLE aux gains d'autrui -> compatible engagement" if plat
               else f"pente non nulle : repond aux gains d'autrui -> sympathie, alpha_hat = {a:.2f}")
    print(f"{nom:<28} {a:9.3f} [{lo:6.3f}, {hi:6.3f}]  {verdict}")
print()
print("La frontiere d'indiscernabilite de GT-06c n'est pas levee par magie : elle est DEPLACEE.")
print("Elle separait deux lectures d'une meme trajectoire (alpha pose par l'analyste) ;")
print("elle se situe desormais au niveau de bruit : alpha_hat dont l'IC95 couvre 0 est non identifie.")

Agent                        alpha_hat               IC95  Verdict
------------------------------------------------------------------------------
Sujet (alpha cache)              0.472 [ 0.439,  0.506]  pente non nulle : repond aux gains d'autrui -> sympathie, alpha_hat = 0.47
Controle sympathie connue        0.820 [ 0.765,  0.872]  pente non nulle : repond aux gains d'autrui -> sympathie, alpha_hat = 0.82
Controle engagement (regle)     -0.004 [-0.042,  0.034]  pente nulle : INSENSIBLE aux gains d'autrui -> compatible engagement

La frontiere d'indiscernabilite de GT-06c n'est pas levee par magie : elle est DEPLACEE.
Elle separait deux lectures d'une meme trajectoire (alpha pose par l'analyste) ;
elle se situe desormais au niveau de bruit : alpha_hat dont l'IC95 couvre 0 est non identifie.


**Lecture de la sortie committée** : le sujet est identifié comme porté par la sympathie (pente non nulle), les contrôles attestent que l'estimateur sait séparer. Mais le gain conceptuel n'est pas « la sympathie a gagné » — c'est le **changement de nature de la frontière** : GT-06c ne pouvait pas trancher parce que son alpha était un choix d'analyste ; ici, ce qui empêche de trancher est un niveau de bruit **mesuré**, réductible par plus de graines ou un horizon plus long. La question passe du « que voulez-vous croire ? » au « combien d'observations faut-il ? ».

### Questions ouvertes — ce que ce dispositif ne modélise pas

- **L'engagement imparfait** : l'agent à règle fixe suit la règle à 95 %. Un engagement réel peut vaciller sous forte tentation — un taux plat à 0,975 sur `R_autre ∈ [0, 6]` ne dit rien de ce qui se passerait à `R_autre = 50`. La statique balaye une plage, pas l'espace entier.
- **La température déclarée** : `beta` est posée (1,0), et la pente ne mesure que le produit `beta * alpha`. L'estimateur hérite de cette convention ; un agent « froid » (grand beta) et un agent très sympathique sont distingués par le contrôle sympathie, pas par la seule pente.
- **Le banc reste un banc** : un adversaire sans mémoire qui coopère toujours est une construction expérimentale, pas un partenaire social. Sen posait sympathie et engagement comme distincts *conceptuellement* ; les séparer *empiriquement* exige une observation que le monde produit rarement aussi proprement que cette grille.

## Exercice 1 : balayer `S_autre` à `R_autre` fixé

La grille de ce notebook fait varier `R_autre` (ce qu'autrui gagne quand je coopère) à `S_autre` fixé. Construisez la statique duale : fixez `R_autre = 3.0` et faites varier `S_autre` sur `[0, 4]`. Avant de simuler, écrivez votre prédiction : le taux doit-il monter ou descendre en `S_autre` ? Vérifiez avec le même estimateur (attention au signe de la pente attendu).

In [7]:
# Exercice 1 a completer
# 1. Construire S_AUTRE_GRID = np.linspace(0.0, 4.0, 9), R_autre fixe a 3.0
# 2. Predire (par ecrit, avant de simuler) le signe de la pente du taux en S_autre
# 3. Mesurer les taux multi-graines du sujet (ALPHA_VRAI) puis ajuster le logit
# Indice : dans le logit, dU = (R - T) + alpha * (R_autre - S_autre) : S_autre entre avec un signe MOINS
print("Exercice a completer")
resultat_ex1 = None  # TODO etudiant : slope_hat_ex1 = ...

Exercice a completer


## Exercice 2 : engagement vacillant sous forte tentation

Le contrôle engagement suit sa règle à 95 % quel que soit `R_autre`. Construisez un engagement **vacillant** : la conformité décroît quand la tentation `T - R` augmente (par ex. `conformite = 0.95 - 0.5 * (R_autre / 6)` — dévier rapporte « moralement » plus quand autrui s'enrichit de ma coopération). L'agent reste-t-il classé « insensible » ? Que mesure alors la pente ?

In [8]:
# Exercice 2 a completer
# 1. taux_engagement_vacillant(R_autre, seed) : conformite = 0.95 - 0.5 * (R_autre / 6.0)
# 2. Mesurer sur la grille, ajuster le logit, comparer alpha_hat au controle rigide
# Indice : la pente sera non nulle — mais NEGATIVE ; est-ce de la sympathie ?
print("Exercice a completer")
resultat_ex2 = None  # TODO etudiant : alpha_hat_vacillant = ...

Exercice a completer


## Exercice 3 : combien de graines pour identifier un petit alpha ?

`ALPHA_VRAI = 0.45` est identifié avec 5 graines × 200 tours. Prenez `ALPHA_VRAI = 0.10` (sympathie faible) et trouvez empiriquement le nombre de graines (parmi 5, 10, 20, 40, graines 0..39, horizon 200) à partir duquel l'IC95 bootstrap exclut 0. C'est la **borne d'indiscernabilité opérationnelle** : le niveau de bruit au-delà duquel la question cesse d'être tranchable.

In [9]:
# Exercice 3 a completer
# 1. Pour chaque effectif de graines dans [5, 10, 20, 40] :
#    mesurer les taux du sujet alpha=0.10, ajuster, bootstrap -> IC95
# 2. Rendre la plus petite effectif dont l'IC95 exclut 0
# Indice : la largeur de l'IC decroit en 1/sqrt(n_decisions) ; le demi-logit a alpha=0.10 est faible
print("Exercice a completer")
resultat_ex3 = None  # TODO etudiant : graines_minimales = ...

Exercice a completer


## Conclusion et perspectives

GT-06c avait isolé la coopération contre-intérêt et posé la bonne question : que la porte-t-il ? Sa limite n'était pas un bug mais une propriété : sur des trajectoires de coopération seules, sympathie et engagement sont **observationnellement équivalents** — le paramètre qui les sépare est choisi par l'analyste, pas mesuré. Ce notebook a déplacé la séparation du côté de l'**expérience** : en faisant varier les gains d'autrui à gains propres fixes, la sympathie devient une **pente** et l'engagement une **absence de pente**, avec des contrôles qui prouvent que l'estimateur sait faire la différence.

Trois acquis :

1. **Identification** : `alpha` se dérive de la pente du logit (`alpha_hat = k / beta`) au lieu d'être posé ; l'incertitude est quantifiée par bootstrap sur les graines.
2. **Validation** : sans le double contrôle négatif (sympathie connue retrouvée, engagement rendu plat), une pente mesurée ne prouverait rien — elle serait un nombre, pas une mesure.
3. **Honnêteté** : le verdict « non identifié à ce niveau de bruit » est un résultat publiable ; choisir un alpha et rapporter le verdict qu'il implique est le geste que cette méthode retire.

Perspectives : le même patron s'applique chaque fois qu'un mécanisme doit être séparé d'un comportement observationnellement équivalent — la statique comparative sur la quantité que **seul** l'un des deux mécanismes consomme. Le voisinage immédiat est #12682 (le partage de Nash : forme posée vs dérivée) et GT-20 (le témoin de retenue : certifier quel mécanisme porte une abstention).